# SGX Multi-Factor — Survivorship-Corrected Backtest (FINAL)Universe: 71 point-in-time FSSTI+SGMCNL names, 2001–2026. Book-to-market from real BVPS (67/71 names). Point-in-time filters: yearly (`Membership`) and quarterly (`MembershipQ`, 92 snapshots 2003Q1–2025Q4).Inputs: `SGX2_v4.xlsm` (PricePanel, Membership, MembershipQ, Classification2), `panel_stock.csv` (survivor list), `macro_monthly.csv` (RF_3M_Ann).**Runs:** A survivor-only (biased) · B PIT-yearly · C PIT-quarterly — full period and post-GFC.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from collections import Counter
DATA = "/mnt/user-data/outputs"
WB = "SGX2_v4.xlsm"

In [2]:
# ---------- 1. Panel (BM from real BVPS) ----------
pp = pd.read_excel(WB, sheet_name="PricePanel")
pp["Date"] = pd.to_datetime(pp["Date"], errors="coerce") + pd.offsets.MonthEnd(0)
pp = pp.dropna(subset=["Date","Ticker"]).sort_values(["Ticker","Date"])
for c in ["PX_LAST","TOT_RETURN","MKT_CAP","BVPS","DIV_YIELD","PX_VOLUME","PE_RATIO","ROE"]:
    pp[c] = pd.to_numeric(pp[c], errors="coerce")
for c in ["ROE","PE_RATIO","BVPS"]:
    pp[c] = pp.groupby("Ticker")[c].ffill()
pp = pp.merge(pd.read_excel(WB, sheet_name="Classification2"), on="Ticker", how="left")
pp["Sector"] = pp["Sector"].fillna("Unknown")

old = pd.read_csv(f"{DATA}/panel_stock.csv", parse_dates=["Date"])
surv = set(old["Ticker"].unique())

s = pp.sort_values(["Ticker","Date"]).reset_index(drop=True)
s["Return_1M"]=s.groupby("Ticker")["TOT_RETURN"].pct_change(1)
s["Ret_PX_1M"]=s.groupby("Ticker")["PX_LAST"].pct_change(1)
s["MOM_12_1"] =s.groupby("Ticker")["PX_LAST"].pct_change(12).shift(1)
s["MOM_6_1"]  =s.groupby("Ticker")["PX_LAST"].pct_change(6).shift(1)
s["MOM_3_1"]  =s.groupby("Ticker")["PX_LAST"].pct_change(3).shift(1)
s["LogMktCap"]=np.log(s["MKT_CAP"].clip(lower=1))
s["BM_Ratio"] =np.where(s["BVPS"]>0, s["BVPS"]/s["PX_LAST"], np.nan)
s["LogVolume"]=np.log(s["PX_VOLUME"].clip(lower=1))
s["Amihud"]   =s["Ret_PX_1M"].abs()/s["PX_VOLUME"].clip(lower=1e6)

mac=pd.read_csv(f"{DATA}/macro_monthly.csv", parse_dates=["Date"]); mac["Date"]=mac["Date"]+pd.offsets.MonthEnd(0)
s=s.merge(mac[["Date","RF_3M_Ann","Regime_RiskOn","Regime_SGGrowth"]].drop_duplicates("Date"), on="Date", how="left")
RF_M=mac.dropna(subset=["RF_3M_Ann"]).drop_duplicates("Date").set_index("Date")["RF_3M_Ann"]/12.0

In [3]:
# ---------- 2. Factors ----------
def cs_z(x):
    mu,sig=x.mean(),x.std(); return (x-mu)/sig if sig>1e-9 else x*0.0
def wins(x,lo=.01,hi=.99): return x.clip(x.quantile(lo),x.quantile(hi))
FS={"MOM_12_1":("MOM_12_1",1),"MOM_6_1":("MOM_6_1",1),"MOM_3_1":("MOM_3_1",1),"VALUE_BM":("BM_Ratio",1),
    "VALUE_DY":("DIV_YIELD",1),"VALUE_PE_INV":("PE_RATIO",-1),"QUALITY_ROE":("ROE",1),
    "SIZE_SMALL":("LogMktCap",-1),"LIQ_AMIHUD":("Amihud",-1)}
W={"Z_MOM_12_1":.20,"Z_MOM_6_1":.05,"Z_VALUE_BM":.20,"Z_VALUE_DY":.10,"Z_VALUE_PE_INV":.10,
   "Z_QUALITY_ROE":.20,"Z_SIZE_SMALL":.05,"Z_LIQ_AMIHUD":.10}
p=s[s["Date"]>="2004-01-01"].copy()
for fn,(col,d) in FS.items():
    raw=p.groupby("Date")[col].transform(wins); p[f"Z_{fn}"]=d*p.groupby("Date")[raw.name].transform(cs_z)
zc=list(W); wv=np.array([W[c] for c in zc]); Z=p[zc]; mask=Z.notna()
num=(Z.mul(wv,axis=1)).sum(axis=1,min_count=1); den=(mask.mul(wv,axis=1)).sum(axis=1)
p["Composite"]=np.where(mask.sum(axis=1)>=4, num/den, np.nan)
p=p.drop_duplicates(subset=["Date","Ticker"]).reset_index(drop=True)

In [4]:
# ---------- 3. Membership filters: yearly vs quarterly ----------
def make_filter(mem_df):
    mem_df = mem_df.copy()
    mem_df["AsOfDate"]=pd.to_datetime(mem_df["AsOfDate"])+pd.offsets.MonthEnd(0)
    snaps=sorted(mem_df["AsOfDate"].unique())
    def f(d):
        valid=[x for x in snaps if x<=d]
        return set(mem_df[mem_df["AsOfDate"]==valid[-1]]["Ticker"]) if valid else None
    return f
am_yearly    = make_filter(pd.read_excel(WB, sheet_name="Membership"))
am_quarterly = make_filter(pd.read_excel(WB, sheet_name="MembershipQ"))

In [5]:
# ---------- 4. Engine ----------
N=6; TC=10
def perf(r,lab):
    rfa=RF_M.reindex(r.index).ffill().fillna(0); ex=r-rfa; c=(1+r).cumprod(); ar=(1+r).prod()**(12/len(r))-1
    rm=c.cummax(); mdd=((c-rm)/rm).min(); dn=ex[ex<0].std()
    return {"Run":lab,"Ann_Return_%":round(ar*100,2),"Ann_Vol_%":round(r.std()*np.sqrt(12)*100,2),
            "Sharpe":round(ex.mean()/ex.std()*np.sqrt(12),3) if ex.std()>0 else 0,
            "Sortino":round(ex.mean()/dn*np.sqrt(12),3) if dn>0 else 0,
            "Max_DD_%":round(mdd*100,2),"Calmar":round(ar/abs(mdd),3) if mdd<0 else 0,"N":len(r)}
def run_bt(panel, member_fn=None, restrict=None, start="2006-01-01", end="2026-01-31"):
    sub=panel[(panel["Date"]>=start)&(panel["Date"]<=end)].copy()
    dts=sorted(sub["Date"].unique()); rows=[]; pl=[]; ps=[]; held=[]
    for i,d in enumerate(dts[:-1]):
        curr=sub[sub["Date"]==d].dropna(subset=["Composite"])
        if restrict is not None: curr=curr[curr["Ticker"].isin(restrict)]
        if member_fn is not None:
            am=member_fn(d)
            if am is not None: curr=curr[curr["Ticker"].isin(am)]
        curr=curr.sort_values("Composite",ascending=False)
        if len(curr)<10: continue
        lt=curr.head(N)["Ticker"].tolist(); st=curr.tail(N)["Ticker"].tolist()
        nd=dts[i+1]; nr=sub[sub["Date"]==nd].drop_duplicates("Ticker").set_index("Ticker")["Return_1M"]
        lg=nr.reindex(lt).mean(); sg=nr.reindex(st).mean()
        lto=len(set(lt)-set(pl))/N if pl else 1.0; sto=len(set(st)-set(ps))/N if ps else 1.0
        ln=lg-lto*TC/1e4; sn=(-sg)-sto*TC/1e4
        rows.append({"Date":nd,"Long":ln,"LS":ln+sn}); held.append((d,lt)); pl=lt; ps=st
    return pd.DataFrame(rows).set_index("Date"), held

runs = {}
runs["A  Survivor-only"],_          = run_bt(p, restrict=surv)
runs["B  PIT yearly"],_             = run_bt(p, member_fn=am_yearly)
runs["C  PIT quarterly"],heldC      = run_bt(p, member_fn=am_quarterly)

print("=== FULL PERIOD 2006-2026 | full BVPS book-to-market ===")
rows=[]
for lab,bt in runs.items():
    rows.append(perf(bt["Long"].dropna(), lab+"  [Long]"))
for lab,bt in runs.items():
    rows.append(perf(bt["LS"].dropna(),   lab+"  [L/S]"))
full=pd.DataFrame(rows); print(full.to_string(index=False))

print("\n=== POST-GFC 2011-2026 ===")
rows=[]
for lab, kw in [("A  Survivor-only",dict(restrict=surv)),("B  PIT yearly",dict(member_fn=am_yearly)),("C  PIT quarterly",dict(member_fn=am_quarterly))]:
    bt,_=run_bt(p, start="2011-01-01", **kw); rows.append(perf(bt["Long"].dropna(),lab+"  [Long]"))
post=pd.DataFrame(rows); print(post.to_string(index=False))

print("\nDelisted names held (quarterly PIT long book):")
cnt=Counter([t for _,lt in heldC for t in lt if t not in surv])
for t,n in cnt.most_common(10): print(f"  {t:18s} {n:3d} months")

full.to_csv(f"{DATA}/survivorship_summary_final.csv", index=False)
p.to_csv(f"{DATA}/panel_with_factors_71_final.csv", index=False)
print("\nSaved survivorship_summary_final.csv + panel_with_factors_71_final.csv")

=== FULL PERIOD 2006-2026 | full BVPS book-to-market ===
                     Run  Ann_Return_%  Ann_Vol_%  Sharpe  Sortino  Max_DD_%  Calmar   N
A  Survivor-only  [Long]         17.37      16.32   1.002    1.540    -44.18   0.393 240
   B  PIT yearly  [Long]          7.82      19.33   0.435    0.506    -67.13   0.116 240
C  PIT quarterly  [Long]          7.91      18.65   0.447    0.543    -62.65   0.126 240
 A  Survivor-only  [L/S]         10.35      17.75   0.586    0.749    -32.45   0.319 240
    B  PIT yearly  [L/S]          2.14      17.32   0.147    0.198    -65.26   0.033 240
 C  PIT quarterly  [L/S]          5.03      16.78   0.313    0.406    -45.48   0.111 240

=== POST-GFC 2011-2026 ===


                     Run  Ann_Return_%  Ann_Vol_%  Sharpe  Sortino  Max_DD_%  Calmar   N
A  Survivor-only  [Long]         14.98      13.00   1.067    1.754    -16.47   0.909 180
   B  PIT yearly  [Long]         10.10      14.49   0.669    1.035    -26.66   0.379 180
C  PIT quarterly  [Long]          9.86      14.14   0.667    1.034    -28.29   0.348 180

Delisted names held (quarterly PIT long book):
  STH SP Equity       73 months
  CD SP Equity        63 months
  FNN SP Equity       49 months
  OLAM SP Equity      42 months
  CCT SP Equity       41 months
  GGR SP Equity       40 months
  SUN SP Equity       25 months
  KPLD SP Equity      24 months
  HPHT SP Equity      24 months
  SPC SP Equity       22 months



Saved survivorship_summary_final.csv + panel_with_factors_71_final.csv
